# 2026 MediaEval NewsImages — Extended with KMeans Clustering

**Prompt template:** `news image thumbnail, illustration, vector art, {headline}`

In [ ]:
# @title packages
!pip install -q diffusers transformers datasets accelerate safetensors \
    sentencepiece open_clip_torch torch torchvision easyocr matplotlib ftfy scikit-learn

In [ ]:
# @title setup
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

import torch
import torch.nn.functional as F
import easyocr
from diffusers import AutoPipelineForText2Image
from transformers import AutoModel, AutoProcessor, CLIPProcessor, CLIPModel

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. SDXL-Turbo (1-step image generator)
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16",
).to(DEVICE)

# 2. OCR — rejects thumbnails that contain rendered text
reader = easyocr.Reader(['en'], gpu=(DEVICE == 'cuda'))

# 3. Standard CLIP (baseline selector)
standard_clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
standard_clip_model     = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
standard_clip_model.eval()

# 4. CFT-CLIP (news-thumbnail-specific selector + embedding source)
cft_clip_processor = AutoProcessor.from_pretrained("humane-lab/CFT-CLIP")
cft_clip_model     = AutoModel.from_pretrained("humane-lab/CFT-CLIP").to(DEVICE)
cft_clip_model.eval()

In [ ]:
# @title pipeline

import re
import ftfy
import numpy as np
from PIL import Image, ImageOps

PROMPT_PREFIX = "news image thumbnail, illustration, vector art, "

# Patterns for trailing publisher/section suffixes to strip
_PIPE_SUFFIX  = re.compile(r'\s*\|.*$')
_DASH_TRAILER = re.compile(
    r'\s*-\s+('
    r'(News|Recap|Roundup|Update|Report|Review|Preview|Podcast|Video|Photo|Photos|Gallery)'
    r'|((January|February|March|April|May|June|July|August|September|October|November|December)\s+\d)'
    r'|(Week|Day|Season|Episode|Vol|No)\s+\d'
    r').*$',
    re.IGNORECASE
)

def clean_title(raw: str) -> str:
    """Fix encoding, strip trailing publisher suffixes, collapse whitespace."""
    t = ftfy.fix_text(str(raw))
    t = _PIPE_SUFFIX.sub('', t)
    t = _DASH_TRAILER.sub('', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t


def sanitize_prompt(raw_text: str) -> str:
    """Strip quoted sub-strings and leading dashes from prompt text."""
    text = re.sub(r'["\'].*?["\']', '', raw_text)
    text = re.sub(r'^.*?\-\s*', '', text)
    return text.strip()


def has_text(image: Image.Image, threshold: float = 0.5) -> bool:
    """Return True if EasyOCR detects high-confidence rendered text in the image."""
    results = reader.readtext(np.array(image))
    return any(prob > threshold and len(t.strip()) > 1 for _, t, prob in results)


def score_standard_clip(image: Image.Image, text: str) -> float:
    """Compute image-text alignment with standard OpenAI CLIP."""
    inputs = standard_clip_processor(
        text=[text], images=image, return_tensors='pt', padding=True
    ).to(DEVICE)
    with torch.no_grad():
        outputs = standard_clip_model(**inputs)
    return outputs.logits_per_image.item()


def score_cft_clip(image: Image.Image, text: str) -> float:
    """Compute image-text cosine similarity with news-specific CFT-CLIP."""
    inputs = cft_clip_processor(
        text=[text], images=image, return_tensors='pt', padding=True
    ).to(DEVICE)
    with torch.no_grad():
        outputs = cft_clip_model(**inputs)
        t = F.normalize(outputs.text_embeds,  p=2, dim=-1)
        i = F.normalize(outputs.image_embeds, p=2, dim=-1)
    return torch.matmul(t, i.t()).item()


def get_image_embedding(image: Image.Image) -> np.ndarray:
    """
    Extract the L2-normalized CFT-CLIP image embedding vector.
    This vector is used as the clustering feature for KMeans.
    Embeddings are already on the unit sphere, so cosine distance == Euclidean.
    Returns a 1-D numpy array of shape (embed_dim,).
    """
    inputs = cft_clip_processor(images=image, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        image_features = cft_clip_model.get_image_features(**inputs)
        # L2-normalize so cosine distance = Euclidean distance on unit sphere
        image_features = F.normalize(image_features, p=2, dim=-1)
    return image_features.cpu().numpy().squeeze()


def generate_thumbnail(headline: str, n: int = 5):
    """
    Generate n candidates for headline with SDXL-Turbo (1-step).
    Rejects candidates that contain visible text (OCR filter).
    Selects the best candidate by CFT-CLIP cosine similarity score.

    Returns
    -------
    best_image     : PIL.Image (460x260)
    best_embedding : np.ndarray  CFT-CLIP image vector of best_image
    best_cft_score : float       CFT-CLIP score of best_image vs headline
    history        : list of dicts with keys image, std_score, cft_score, rejected
    """
    clean  = sanitize_prompt(headline)
    prompt = PROMPT_PREFIX + clean

    best_image, best_embedding, best_cft, fallback = None, None, -float('inf'), None
    history = []

    for _ in range(n):
        out = pipe(
            prompt=prompt,
            num_inference_steps=1,
            guidance_scale=0.0,
            height=512,
            width=512,
        )
        img = ImageOps.fit(out.images[0], (460, 260), method=Image.Resampling.LANCZOS)
        if fallback is None:
            fallback = img

        rejected  = has_text(img)
        std_score = cft_score = -1.0
        embedding = None

        if not rejected:
            std_score = score_standard_clip(img, clean)
            cft_score = score_cft_clip(img, clean)
            embedding = get_image_embedding(img)
            if cft_score > best_cft:
                best_cft       = cft_score
                best_image     = img
                best_embedding = embedding

        history.append({
            'image':     img,
            'std_score': std_score,
            'cft_score': cft_score,
            'rejected':  rejected,
        })

    # Fallback: if all candidates were rejected, use the first generated image
    if best_image is None:
        best_image     = fallback
        best_embedding = get_image_embedding(fallback)
        best_cft       = score_cft_clip(fallback, clean)

    return best_image, best_embedding, best_cft, history

In [ ]:
# @title eval (single-article sanity check)

import matplotlib.pyplot as plt

N        = 20
headline = "The central bank decided to lower interest rates by 0.5% today."

best, best_emb, best_score, candidates = generate_thumbnail(headline, n=N)
best.save("output_thumbnail.png", "PNG")

# Score summary
valid = [c for c in candidates if not c['rejected']]
if valid:
    print(f'Valid candidates : {len(valid)}/{N}')
    print(f"CFT-CLIP range   : [{min(c['cft_score'] for c in valid):.4f}, "
          f"{max(c['cft_score'] for c in valid):.4f}]")
    print(f"CLIP range       : [{min(c['std_score'] for c in valid):.4f}, "
          f"{max(c['std_score'] for c in valid):.4f}]")
    print(f'Best CFT-CLIP score: {best_score:.4f}')
    print(f'Embedding shape    : {best_emb.shape}')
else:
    print('No valid candidates (all rejected by OCR filter).')

# Candidate grid visualization
cols = 4
rows = (N + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
axes = axes.flatten()

for i, c in enumerate(candidates):
    ax = axes[i]
    ax.imshow(c['image'])
    ax.axis('off')
    if c['rejected']:
        ax.set_title(f'#{i+1} REJECTED', color='red', fontsize=8, fontweight='bold')
    else:
        selected = (c['image'] is best)
        label    = f"#{i+1}  CLIP={c['std_score']:.3f}\nCFT={c['cft_score']:.3f}"
        if selected:
            label += '\n[SELECTED]'
        ax.set_title(label,
                     color='green' if selected else 'black',
                     fontsize=8,
                     fontweight='bold' if selected else 'normal')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle(headline, fontsize=10)
plt.tight_layout()
plt.savefig("candidate_grid.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# @title Exp 1: CFT-CLIP vs Standard CLIP as selector

import matplotlib.pyplot as plt

ARTICLES = [
    "Central bank cuts interest rates by 0.5 percent",
    "Wildfire spreads across northern California destroying hundreds of homes",
    "Tech giant announces layoffs affecting 10000 employees",
    "Scientists discover new species of deep sea fish near Pacific trench",
    "Olympic gold medalist retires after twenty year career",
]
N_POOL = 20

exp1 = []
for headline in ARTICLES:
    _, _, _, pool = generate_thumbnail(headline, n=N_POOL)
    valid    = [c for c in pool if not c['rejected']]
    fallback = pool[0]
    best_clip = max(valid, key=lambda c: c['std_score']) if valid else fallback
    best_cft  = max(valid, key=lambda c: c['cft_score']) if valid else fallback
    exp1.append({'headline': headline, 'clip': best_clip, 'cft': best_cft})

fig, axes = plt.subplots(len(ARTICLES), 2, figsize=(10, 4 * len(ARTICLES)))
for i, r in enumerate(exp1):
    for j, (key, label, color) in enumerate([
        ('clip', 'Best by CLIP',     'steelblue'),
        ('cft',  'Best by CFT-CLIP', 'darkorange'),
    ]):
        ax = axes[i][j]
        ax.imshow(r[key]['image'])
        ax.axis('off')
        ax.set_title(
            f"{label}\nCLIP={r[key]['std_score']:.3f}  CFT={r[key]['cft_score']:.3f}"
            f"\n{r['headline'][:55]}",
            fontsize=8, color=color,
        )

plt.tight_layout()
plt.savefig("exp1_clip_vs_cft.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved exp1_clip_vs_cft.png")

In [ ]:
# @title Exp 2: Best-of-N scaling  (N = 1, 5, 25, 125)

import matplotlib.pyplot as plt
import numpy as np

ARTICLES = [
    "Central bank cuts interest rates by 0.5 percent",
    "Wildfire spreads across northern California destroying hundreds of homes",
    "Tech giant announces layoffs affecting 10000 employees",
    "Scientists discover new species of deep sea fish near Pacific trench",
    "Olympic gold medalist retires after twenty year career",
]
N_MAX    = 125
N_VALUES = [1, 5, 25, 125]

# Generate the full candidate pool once per article (re-use for all N)
pools = {}
for headline in ARTICLES:
    _, _, _, pool = generate_thumbnail(headline, n=N_MAX)
    pools[headline] = pool
    valid_count = sum(1 for c in pool if not c['rejected'])
    print(f'  {headline[:50]:50s}  valid={valid_count}/{N_MAX}')


def best_cft_of_n(pool, n):
    """Return the highest CFT-CLIP score from the first n valid candidates."""
    valid  = [c for c in pool if not c['rejected']]
    subset = valid[:n] if len(valid) >= n else valid
    return max((c['cft_score'] for c in subset), default=-1.0)


mean_scores = [
    np.mean([best_cft_of_n(pools[h], n) for h in ARTICLES])
    for n in N_VALUES
]

# Scaling curve
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(N_VALUES, mean_scores, marker='o', color='darkorange', linewidth=2)
ax.set_xscale('log')
ax.set_xticks(N_VALUES)
ax.set_xticklabels([str(n) for n in N_VALUES])
ax.set_xlabel('N (candidates sampled)')
ax.set_ylabel('Mean CFT-CLIP Score')
ax.set_title('Best-of-N Scaling')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("exp2_best_of_n_curve.png", dpi=150, bbox_inches="tight")
plt.show()

# N=1 vs N=125 side-by-side for first article
h0     = ARTICLES[0]
valid0 = [c for c in pools[h0] if not c['rejected']]
img_n1   = valid0[0]['image'] if valid0 else pools[h0][0]['image']
img_n125 = max(valid0, key=lambda c: c['cft_score'])['image'] if valid0 else pools[h0][0]['image']

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, img, label in zip(axes, [img_n1, img_n125], ['N=1 (random)', 'N=125 (best-of)']):
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(label)
plt.suptitle(h0, fontsize=9)
plt.tight_layout()
plt.savefig("exp2_n1_vs_n125.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved exp2_best_of_n_curve.png, exp2_n1_vs_n125.png")

# Final Run + KMeans Clustering (K=27)

以下三個 Cell 完成：
1. 對所有測試文章生成 best-of-5 縮圖並提取 CFT-CLIP 影像向量
2. 以 KMeans (K=27) 對影像向量進行聚類
3. 視覺化聚類結果（PCA 散佈圖、分數長條圖、代表縮圖格）

In [ ]:
# @title Step 1 — Generate best-of-5 thumbnails for all test articles

import os
import pandas as pd
import numpy as np

N_CANDIDATES = 5          # best-of-N per article (as required)
OUT_DIR      = "thumbnails_best"
os.makedirs(OUT_DIR, exist_ok=True)

# Load test CSV - update filename path if needed in Colab
df_test = pd.read_csv("news_articles_test.csv", encoding="latin1")
print(f'Loaded {len(df_test)} articles from news_articles_test.csv')

# Accumulators for downstream clustering
article_ids     = []   # article ID per row
headlines_clean = []   # cleaned headline text
best_images     = []   # best PIL image per article
embeddings      = []   # CFT-CLIP image embedding vector per article
cft_scores      = []   # best-of-5 CFT-CLIP score per article

for idx, row in df_test.iterrows():
    article_id = row['article_id']
    raw_title  = row['article_title']
    headline   = clean_title(raw_title)

    print(f"[{idx+1}/{len(df_test)}] ID={article_id}  '{headline[:60]}'")

    # Generate 5 candidates; pick best by CFT-CLIP; get its embedding vector
    best_img, best_emb, best_score, _ = generate_thumbnail(headline, n=N_CANDIDATES)

    # Save the selected thumbnail to disk
    fname = f'{article_id}.png'
    best_img.save(os.path.join(OUT_DIR, fname), 'PNG')

    article_ids.append(article_id)
    headlines_clean.append(headline)
    best_images.append(best_img)
    embeddings.append(best_emb)
    cft_scores.append(best_score)

# Stack embeddings into matrix shape: (N_articles, embed_dim)
embedding_matrix = np.stack(embeddings, axis=0)

print(f'\nDone! Generated {len(best_images)} thumbnails.')
print(f'Embedding matrix shape : {embedding_matrix.shape}')
print(f'CFT-CLIP score range   : [{min(cft_scores):.4f}, {max(cft_scores):.4f}]')

In [ ]:
# @title Step 2 — KMeans clustering on CFT-CLIP image embeddings (K=27)

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

K = 27   # number of clusters

# Embeddings are L2-normalized; Euclidean distance approximates cosine distance
kmeans = KMeans(
    n_clusters=K,
    init='k-means++',   # smarter init to speed up convergence
    n_init=10,           # run 10 restarts, keep the best
    max_iter=300,
    random_state=42,
)
cluster_labels = kmeans.fit_predict(embedding_matrix)

# Silhouette score: measures intra-cluster cohesion vs inter-cluster separation
# Range [-1, 1]; higher is better
sil_score = silhouette_score(embedding_matrix, cluster_labels, metric='cosine')
print(f'KMeans converged in {kmeans.n_iter_} iterations')
print(f'Inertia (within-cluster SSE) : {kmeans.inertia_:.4f}')
print(f'Silhouette score (cosine)    : {sil_score:.4f}')

# Compute per-cluster statistics (size and mean/std CFT-CLIP score)
cluster_sizes       = []
cluster_scores_mean = []
cluster_scores_std  = []
cluster_labels_list = cluster_labels.tolist()

print(f"\n{'Cluster':>8}  {'Size':>6}  {'Mean CFT':>10}  {'Std CFT':>9}")
print('-' * 42)
for k in range(K):
    mask   = (cluster_labels == k)
    scores = [cft_scores[i] for i in range(len(cft_scores)) if mask[i]]
    size   = int(mask.sum())
    mean_s = float(np.mean(scores)) if scores else 0.0
    std_s  = float(np.std(scores))  if scores else 0.0
    cluster_sizes.append(size)
    cluster_scores_mean.append(mean_s)
    cluster_scores_std.append(std_s)
    print(f'{k:>8}  {size:>6}  {mean_s:>10.4f}  {std_s:>9.4f}')

In [ ]:
# @title Step 3 — Visualize KMeans clusters (PCA 2-D projection)

import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

# Reduce embedding dimensions to 2-D for 2D scatter plot
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(embedding_matrix)

explained = pca.explained_variance_ratio_ * 100
print(f'PCA variance explained: PC1={explained[0]:.1f}%  PC2={explained[1]:.1f}%  '
      f'Total={sum(explained):.1f}%')

# Project cluster centroids to 2D as well
centroid_2d = pca.transform(kmeans.cluster_centers_)

cmap = plt.cm.get_cmap('tab20', K)

fig, ax = plt.subplots(figsize=(14, 9))
sc = ax.scatter(
    coords_2d[:, 0], coords_2d[:, 1],
    c=cluster_labels,
    cmap='tab20',
    s=40,
    alpha=0.75,
    edgecolors='none',
)

# Annotate each centroid with its cluster ID
for k in range(K):
    ax.text(
        centroid_2d[k, 0], centroid_2d[k, 1],
        str(k),
        fontsize=9, fontweight='bold',
        ha='center', va='center',
        color='white',
        bbox=dict(boxstyle='round,pad=0.2', facecolor=cmap(k), alpha=0.9, edgecolor='none'),
    )

plt.colorbar(sc, ax=ax, label='Cluster ID')
ax.set_xlabel(f'PC 1 ({explained[0]:.1f}% variance)')
ax.set_ylabel(f'PC 2 ({explained[1]:.1f}% variance)')
ax.set_title(f'KMeans Clustering (K={K}) of CFT-CLIP Image Embeddings\n'
             f'PCA 2-D Projection  |  {len(embedding_matrix)} articles  '
             f'|  Silhouette={sil_score:.4f}')
ax.grid(True, alpha=0.2, linestyle='--')
plt.tight_layout()
plt.savefig("kmeans_pca_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved kmeans_pca_scatter.png")

In [ ]:
# @title Step 4 — Cluster score bar chart

import matplotlib.pyplot as plt
import numpy as np

# Sort clusters by descending mean CFT-CLIP score for easier comparison
sorted_idx = np.argsort(cluster_scores_mean)[::-1]
global_mean = np.mean(cft_scores)

fig, axes = plt.subplots(2, 1, figsize=(16, 9))

# Top panel: mean CFT-CLIP score per cluster with std-dev error bars
ax = axes[0]
x  = np.arange(K)
ax.bar(
    x,
    [cluster_scores_mean[i] for i in sorted_idx],
    yerr=[cluster_scores_std[i] for i in sorted_idx],
    color=[plt.cm.tab20(i / K) for i in range(K)],
    capsize=3, edgecolor='none', alpha=0.85,
)
ax.set_xticks(x)
ax.set_xticklabels([str(i) for i in sorted_idx], fontsize=8)
ax.set_xlabel('Cluster ID (sorted by mean CFT-CLIP score)')
ax.set_ylabel('Mean CFT-CLIP Score')
ax.set_title(f'Cluster Quality: Mean CFT-CLIP Score per Cluster  (K={K})')
ax.axhline(global_mean, color='red', linestyle='--', linewidth=1, label='Global mean')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Bottom panel: number of articles per cluster
ax2 = axes[1]
ax2.bar(
    x,
    [cluster_sizes[i] for i in sorted_idx],
    color=[plt.cm.tab20(i / K) for i in range(K)],
    edgecolor='none', alpha=0.85,
)
ax2.set_xticks(x)
ax2.set_xticklabels([str(i) for i in sorted_idx], fontsize=8)
ax2.set_xlabel('Cluster ID (sorted by mean CFT-CLIP score)')
ax2.set_ylabel('Number of Articles')
ax2.set_title('Cluster Size Distribution')
ax2.axhline(len(embedding_matrix) / K, color='red', linestyle='--', linewidth=1,
            label=f'Expected uniform size ({len(embedding_matrix)//K})')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("kmeans_cluster_scores.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved kmeans_cluster_scores.png")

In [ ]:
# @title Step 5 — Representative thumbnail grid per cluster

import matplotlib.pyplot as plt
import numpy as np

# For each cluster: find the article whose embedding is closest to the centroid
representative_idx = []
for k in range(K):
    mask = np.where(cluster_labels == k)[0]
    if len(mask) == 0:
        representative_idx.append(None)
        continue
    centroid = kmeans.cluster_centers_[k]
    dists    = np.linalg.norm(embedding_matrix[mask] - centroid, axis=1)
    rep      = mask[np.argmin(dists)]
    representative_idx.append(int(rep))

# Layout: 5 columns
COLS = 5
ROWS = (K + COLS - 1) // COLS
fig  = plt.figure(figsize=(COLS * 3.5, ROWS * 3.2))

for plot_pos in range(K):
    ax    = fig.add_subplot(ROWS, COLS, plot_pos + 1)
    rep_i = representative_idx[plot_pos]
    if rep_i is not None:
        ax.imshow(best_images[rep_i])
        title = (f'Cluster {plot_pos}\n'
                 f'n={cluster_sizes[plot_pos]}  CFT={cluster_scores_mean[plot_pos]:.3f}\n'
                 f'{headlines_clean[rep_i][:40]}')
        ax.set_title(title, fontsize=6.5, pad=3)
    else:
        ax.set_title(f'Cluster {plot_pos}\n(empty)', fontsize=7)
    ax.axis('off')

plt.suptitle(
    f'Representative Thumbnail per KMeans Cluster (K={K})  —  CFT-CLIP Embeddings',
    fontsize=11, y=1.01,
)
plt.tight_layout()
plt.savefig("kmeans_cluster_grid.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved kmeans_cluster_grid.png")

In [ ]:
# @title Step 6 — Save results CSV + submission ZIP

import os, zipfile
import pandas as pd

GROUP_NAME    = 'TEAM_A++'
APPROACH_NAME = 'sdxl_cft_bon5_kmeans27'

# Full results dataframe: article + cluster assignment + scores
results_df = pd.DataFrame({
    'article_id':        article_ids,
    'headline':          headlines_clean,
    'cft_score':         cft_scores,
    'cluster_label':     cluster_labels_list,
    'cluster_mean_cft':  [cluster_scores_mean[c] for c in cluster_labels_list],
    'cluster_size':      [cluster_sizes[c] for c in cluster_labels_list],
    'thumbnail_path':    [os.path.join('thumbnails_best', f'{aid}.png') for aid in article_ids],
})
results_df.to_csv('clustering_results.csv', index=False)
print('Saved clustering_results.csv')
print(results_df.head(10).to_string())

# Package into submission ZIP
ZIP_PATH = f'{GROUP_NAME}_Submission.zip'
with zipfile.ZipFile(ZIP_PATH, 'w') as zf:
    for fname in sorted(os.listdir('thumbnails_best')):
        zf.write(os.path.join('thumbnails_best', fname),
                 arcname=f'thumbnails_best/{fname}')
    for extra in ['clustering_results.csv',
                  'kmeans_pca_scatter.png',
                  'kmeans_cluster_scores.png',
                  'kmeans_cluster_grid.png']:
        if os.path.exists(extra):
            zf.write(extra)

print(f'Zipped {len(os.listdir("thumbnails_best"))} thumbnails + cluster outputs -> {ZIP_PATH}')

# Download in Colab
from google.colab import files
files.download(ZIP_PATH)